# Countries Scale-Up - Walkthrough

This notebook walks through the CR-Box / coal-baghouse scale-up pipeline that lives in
`src/countries.py`. The single big idea is:

* For each country we add up four streams of clean-air-delivery-rate (CADR) over time:
  1. **CR Box weekly manufacturing** (with a small distribution delay)
  2. **CR Box repurposing** of existing HVAC capacity
  3. **CR Box initial stock** (immediate ramp at week `Initial_Stock_Delay`)
  4. **Coal-baghouse retrofits**
* The sum is the cumulative CADR delivered to that country each week.
* We compare that to the country's *indoor essential* / *indoor vital* worker population
  to get the time-to-reach the WHO 5-ACH benchmark (`CADRPP`).

Inputs:

| File | Provides |
| --- | --- |
| `data/scale_up/STANDARD_COUNTRY_LIST.csv` | ISO-3 ↔ short name ↔ long name. |
| `data/scale_up/CR_Box_Countries_MS.csv` | Big-6 manufacturing flags (MSA), Manufacturing Value Added (MVA), Manufacturing Friction Score (MFS). |
| `data/scale_up/BaghouseAirflow.csv` | Coal-baghouse operating MW per country. |
| `results/essential_workers/EssentialWorkersByCountry.csv` | Output of `Essential_Worker_Processing.ipynb`. |

Outputs (written to `results/scale_up/`):

* `Scale_up_output_MS.csv` / `.pkl` - the main CADR trajectory by country & region.
* `Scale_up_percent_indoor_vital.csv` / `.pkl` - the %-of-population-covered trajectory.
* `Scale_up_TTR.csv` (or similar) - the time-to-reach tables.
* Per-stream tables for repurposing, manufacturing, initial stock, coal baghouse.

## 0. Setup

This notebook depends on `Essential_Worker_Processing.ipynb` having been run first
(it uses `results/essential_workers/EssentialWorkersByCountry.csv`).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
sys.path.insert(0, str(REPO / 'src'))

import countries as cc
import essential_workers as ew

EW_DATA = REPO / 'data' / 'essential_workers'
EW_RESULTS = REPO / 'results' / 'essential_workers'
SCALE_UP_DATA = REPO / 'data' / 'scale_up'
SCALE_UP_RESULTS = REPO / 'results' / 'scale_up'
EW_RESULTS.mkdir(parents=True, exist_ok=True)
SCALE_UP_RESULTS.mkdir(parents=True, exist_ok=True)

if not (EW_RESULTS / 'EssentialWorkersByCountry.csv').exists():
    print('EssentialWorkersByCountry.csv missing - running EW pipeline first…')
    ew.run_pipeline(data_dir=EW_DATA, results_dir=EW_RESULTS, write=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

## 1. Quick-start: run the whole pipeline

A single call materialises every table the project cares about; the rest of the notebook
shows the intermediate stages and how the constants in `countries.py` shape them.

In [2]:
out = cc.run_pipeline(data_dir=SCALE_UP_DATA, results_dir=SCALE_UP_RESULTS, write=True)
print(f'Countries  : {len(out.countries)}')
print(f'Main df    : {out.main_df.shape}')
print(f'Region TTR : {out.ttr_region_df.shape}')
out.pct_1y_region_df

/home/james/miniforge3/envs/InRoomAirFilterScaleUp/lib/python3.11/site-packages/uncertainties/core.py:1024: UserWarning: Using UFloat objects with std_dev==0 may give unexpected results.
  warn("Using UFloat objects with std_dev==0 may give unexpected results.")


Countries  : 216
Main df    : (239, 55)
Region TTR : (22, 2)


,Percentage after 1 Year
Region,
Australia and New Zealand,60.681038
Caribbean,12.557405
Central America,38.863185
Central Asia,11.831088
Eastern Africa,0.038914
Eastern Asia,64.358568
Eastern Europe,29.243773
Melanesia,0.328948
Micronesia,2.391282


## 2. Stage-by-stage deep dive

### 2.1 Build the country objects

`generate_countries_from_multiple_csvs` reads the four CSVs and produces a `{ISO-3:
Country}` dict. Each `Country` carries a `properties` dict that the rest of the pipeline
fills in stage by stage.

In [3]:
countries = cc.generate_countries_from_multiple_csvs(
    SCALE_UP_DATA / 'STANDARD_COUNTRY_LIST.csv',
    SCALE_UP_DATA / 'CR_Box_Countries_MS.csv',
    EW_RESULTS / 'EssentialWorkersByCountry.csv',
    SCALE_UP_DATA / 'BaghouseAirflow.csv',
)
print(f'Built {len(countries)} countries')
print('Example - USA properties:')
for k, v in list(countries['USA'].properties.items())[:12]:
    print(f'  {k:35s}  {v}')

Built 216 countries
Example - USA properties:
  ISO-3                                USA
  MFS                                  94.4
  MVA                                  2497130000000.0
  MSA                                  1.0
  Labour Force (2024)                  174000000.0
  Region                               Northern America
  %Indoor Essential Workers            0.2021362510688838
  %Indoor Vital Workers                0.0854906896423872
  %Essential Workers                   0.3089553444585667
  %Vital Workers                       0.144290904802498
  %Armed Forces (Indoor Essential)     0.0
  %Armed Forces (Essential)            0.0


### 2.2 Compute derived per-country properties

`compute_country_properties` writes the CADR-per-week, distribution-delay, etc. for
every Country. Countries lacking any of the required inputs are returned in the
`dropped` list and removed from the dict.

In [4]:
dropped = cc.compute_country_properties(countries)
if dropped:
    print(f'Dropped {len(dropped)} countries missing inputs: {dropped[:10]}{"…" if len(dropped)>10 else ""}')

rows = []
for iso, c in countries.items():
    p = c.properties
    rows.append({
        'ISO': iso,
        'Big_6': p['Big_6'],
        'MFS': p['MFS'],
        'Mfg Delay (wk)': p['CR Box Manufacturing Distribution Delay'],
        'CADR/wk (CR Man)': p['CADR: CR Box Weekly Production'],
        'CADR (CR Repur)':  p['CADR: CR Box Repurposing'],
        'CADR (Initial Stock)': p['CADR: CR Box Initial Stock'],
        'CADR (Coal Baghouse)': p['CADR: Coal Baghouse'],
    })
pd.DataFrame(rows).head(15)

/home/james/miniforge3/envs/InRoomAirFilterScaleUp/lib/python3.11/site-packages/uncertainties/core.py:1024: UserWarning: Using UFloat objects with std_dev==0 may give unexpected results.
  warn("Using UFloat objects with std_dev==0 may give unexpected results.")


,ISO,Big_6,MFS,Mfg Delay (wk),CADR/wk (CR Man),CADR (CR Repur),CADR (Initial Stock),CADR (Coal Baghouse)
0,ABW,False,0.00,0,0.0+/-0,0.0+/-0,0.0+/-0,(6.6+/-2.3)e+03
1,AFG,False,41.00,0,0.0+/-0,(6.4+/-3.4)e+04,0.0+/-0,(6.6+/-2.3)e+03
2,AGO,False,43.25,0,0.0+/-0,0.0+/-0,0.0+/-0,(6.6+/-2.3)e+03
3,ALB,False,41.70,0,0.0+/-0,(8+/-4)e+04,0.0+/-0,(6.6+/-2.3)e+03
4,AND,False,81.00,3,0.0+/-0,(6.4+/-3.4)e+03,0.0+/-0,(6.6+/-2.3)e+03
5,ARE,False,72.50,5,(7.4+/-1.1)e+05,(2.8+/-1.5)e+06,0.0+/-0,(6.6+/-2.3)e+03
6,ARG,False,62.25,7,(1.20+/-0.18)e+06,(5.1+/-2.7)e+06,0.0+/-0,(1.8+/-0.7)e+05
7,ARM,False,50.00,0,0.0+/-0,(1.2+/-0.6)e+05,0.0+/-0,(6.6+/-2.3)e+03
8,ASM,False,37.00,0,0.0+/-0,0.0+/-0,0.0+/-0,(6.6+/-2.3)e+03
9,ATG,False,45.80,0,0.0+/-0,(2.4+/-1.3)e+03,0.0+/-0,(6.6+/-2.3)e+03


### 2.3 Single-country scale-up

Each of the four streams returns a list of length `weeks+1`. `scale_up_MAIN` returns the
elementwise sum. Below we render the USA's first year as an example.

In [5]:
usa = countries['USA']
weeks = 52
df = pd.DataFrame({
    'CR Box Manufacturing': cc.scale_up_CR_MAN(usa, weeks),
    'CR Box Repurposing':   cc.scale_up_CR_REPUR(usa, weeks),
    'CR Box Initial Stock': cc.scale_up_CR_STOCK(usa, weeks),
    'Coal Baghouse':        cc.scale_up_COALBAG(usa, weeks),
    'Total':                cc.scale_up_MAIN(usa, weeks),
})
df.index.name = 'Week'
df.head(15).map(lambda x: getattr(x, 'nominal_value', x))

,CR Box Manufacturing,CR Box Repurposing,CR Box Initial Stock,Coal Baghouse,Total
Week,,,,,
0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
1,3.116655e+07,4.937614e+06,0.000000e+00,0.000000e+00,3.610417e+07
2,6.455929e+07,1.110963e+07,1.869993e+08,0.000000e+00,2.626683e+08
3,1.001782e+08,1.851605e+07,1.869993e+08,0.000000e+00,3.056936e+08
4,1.380233e+08,2.962569e+07,1.869993e+08,6.358234e+07,4.182307e+08
5,1.780946e+08,4.567293e+07,1.869993e+08,6.358234e+07,4.743492e+08
6,2.203921e+08,7.776743e+07,1.869993e+08,6.358234e+07,5.487412e+08
7,2.649157e+08,9.381467e+07,1.869993e+08,6.358234e+07,6.093121e+08
8,3.094394e+08,1.049243e+08,1.869993e+08,6.358234e+07,6.649453e+08


### 2.4 Aggregate every country into the global / regional tables

In [6]:
tables = cc.scale_up_all_countries(countries, weeks=52)
main_df = pd.DataFrame({**tables.country_main, **tables.region_main}).T
print('Per-country + region rows:', main_df.shape[0])
main_df.loc[['Global', 'Northern America', 'Eastern Asia', 'Western Europe'], main_df.columns[:7]].map(
    lambda x: f'{getattr(x, "nominal_value", x):.3e}'
)

Per-country + region rows: 239


,0,1,2,3,4,5,6
Global,4.264e+08,8.940e+08,0.000e+00,1.514e+08,1.034e+09,1.244e+09,2.214e+09
Northern America,1.684e+07,3.982e+07,0.000e+00,3.640e+07,2.657e+08,3.116e+08,4.282e+08
Eastern Asia,9.978e+07,2.161e+08,0.000e+00,7.961e+07,5.771e+08,6.785e+08,1.226e+09
Western Europe,8.840e+06,2.006e+07,0.000e+00,1.875e+07,1.264e+08,1.508e+08,1.901e+08


### 2.5 % of indoor-vital population covered

In [7]:
pct_df = pd.DataFrame({**tables.country_percent_indoor_vital, **tables.region_percent_indoor_vital}).T
pct_df.loc[['Global', 'Northern America', 'Eastern Asia', 'Eastern Africa', 'Western Europe'], pct_df.columns[2:]].map(
    lambda x: f'{getattr(x, "nominal_value", x):.2%}'
).iloc[:, [0, 12, 25, 51]]

,2,14,27,53
Global,0.00%,1097.66%,1843.69%,3335.77%
Northern America,0.00%,5321.94%,8943.26%,16185.91%
Eastern Asia,0.00%,2263.02%,3619.19%,6331.54%
Eastern Africa,0.00%,3.89%,3.89%,3.89%
Western Europe,0.00%,5083.54%,8857.13%,16404.29%


### 2.6 Time-to-reach the 5-ACH benchmark

`time_to_reach` reports the first week each country's cumulative CADR exceeds the
target for its **indoor vital** and **indoor essential** populations
respectively. The horizon is 5 years; countries that don't reach are reported as
`ttr_weeks` (i.e. 261).

In [8]:
ttr_country, ttr_region = cc.time_to_reach(countries, weeks=52*5)
ttr_country_df = pd.DataFrame.from_dict(
    ttr_country, orient='index',
    columns=['Indoor Vital in Weeks', 'Indoor Essential in Weeks'],
)
ttr_region_df = pd.DataFrame.from_dict(
    ttr_region, orient='index',
    columns=['Indoor Vital in Weeks', 'Indoor Essential in Weeks'],
)
print('Top 10 fastest indoor-vital coverage:')
print(ttr_country_df.sort_values('Indoor Vital in Weeks').head(10))
print('\nRegional TTR:')
ttr_region_df

Top 10 fastest indoor-vital coverage:
               Indoor Vital in Weeks  Indoor Essential in Weeks
Singapore                          7                         12
Liechtenstein                      8                         16
Ireland                           18                         39
Germany                           22                         54
Switzerland                       26                         61
United States                     28                         73
Denmark                           31                         78
Japan                             31                         97
Israel                            32                         82
Sweden                            36                         83

Regional TTR:


,Indoor Vital in Weeks,Indoor Essential in Weeks
Caribbean,261,261
Southern Asia,261,261
Middle Africa,261,261
Southern Europe,68,173
Western Asia,238,261
South America,261,261
Polynesia,261,261
Australia and New Zealand,91,227
Western Europe,30,74
Eastern Africa,261,261


### 2.7 % of indoor-vital workforce covered after one year

In [9]:
out.pct_1y_region_df.sort_values('Percentage after 1 Year', ascending=False)

,Percentage after 1 Year
Region,
Western Europe,166.945656
Northern America,164.644712
Northern Europe,122.306716
Southern Europe,78.284620
Eastern Asia,64.358568
Australia and New Zealand,60.681038
Central America,38.863185
Eastern Europe,29.243773
Southern Africa,25.169822


## 3. Re-run the test suite

Run the fast subset with `pytest`, and the slow full-data integration test with
`pytest --full-data`.